# Detector perro vs no-perro — **TensorFlow** (CNN desde cero)

Ejecuta en orden (o **Run All**): dependencias → dataset → entrenamiento → resultados.


In [ ]:
# 1.0) Preparar CUDA para TensorFlow antes de importar el framework
import os
from pathlib import Path

venv_root = Path('/mnt/c/Users/gotts/AI-Frameworks/venv')
nvidia_root = venv_root / 'lib/python3.12/site-packages/nvidia'
if nvidia_root.exists():
    lib_paths = [str(path) for path in nvidia_root.glob('*/lib')]
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    merged = ':'.join(lib_paths + ['/usr/lib/wsl/lib'] + ([current_ld] if current_ld else []))
    os.environ['LD_LIBRARY_PATH'] = merged
    print('LD_LIBRARY_PATH configurado para TensorFlow GPU')
else:
    print('No se encontró', nvidia_root)


In [ ]:
# 1) Dependencias (si falta algo, descomenta el pip)
# %pip install -r requirements.txt
import importlib
for pkg in ['tensorflow', 'numpy', 'matplotlib', 'PIL', 'tqdm']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}: {getattr(mod, "__version__", "ok")}')

In [ ]:
# 2) Dataset (idempotente: si ya está, no re-descarga)
import dataset
dataset.build()

In [ ]:
# 3) Entrenamiento de la CNN desde cero (TensorFlow)
import train_tf
results = train_tf.run(epochs=30)

In [ ]:
# 4) Resultados
print('Clases:', results['class_names'])
print('Matriz de confusión:'); print(results['confusion_matrix'])
print(f"Accuracy: {results['accuracy']:.4f}")
from IPython.display import Image
Image(filename='artifacts/dog_detector_tf_history.png')

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

confusion_matrix = results['confusion_matrix']
class_names = results['class_names']
confusion_matrix_path = Path.cwd() / 'artifacts' / 'dog_detector_tf_confusion_matrix.png'
confusion_matrix_path.parent.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(confusion_matrix, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión')
for i in range(confusion_matrix.shape[0]):
    for j in range(confusion_matrix.shape[1]):
        ax.text(j, i, str(confusion_matrix[i, j]), ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(confusion_matrix_path, dpi=120)
plt.show()
plt.close(fig)
print('Matriz de confusión guardada en', confusion_matrix_path)

In [ ]:
# 5) Inferencia sobre una imagen nueva
from pathlib import Path
import tensorflow as tf
import train_tf

inference_model = tf.keras.models.load_model(train_tf.MODEL_PATH)

def predict_image(image_path, class_names=None):
    image_path = Path(image_path)
    image = tf.keras.utils.load_img(image_path, target_size=train_tf.CONFIG.image_size)
    image_array = tf.keras.utils.img_to_array(image)
    image_array = tf.expand_dims(image_array, axis=0) / 255.0

    probability = float(inference_model.predict(image_array, verbose=0).ravel()[0])
    label_index = int(probability >= 0.5)
    if class_names is None:
        class_names = results['class_names'] if 'results' in globals() else ['not_dog', 'dog']
    label = class_names[label_index]
    print(f'Imagen: {image_path}')
    print(f'Probabilidad de dog: {probability:.4f}')
    print(f'Resultado: {label}')
    return {'image_path': str(image_path), 'prob_dog': probability, 'label': label}

# Cambia esta ruta por tu foto
sample_image_path = 'ruta/a/tu/foto.jpg'
# predict_image(sample_image_path)